In [79]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
import re
import string
from bs4 import BeautifulSoup

In [80]:
df = pd.read_csv('./winter_project_2026/development.csv')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)


# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
#text = df['title'] + ' ' + df['title'] + ' ' + df['article']
#df = pd.concat([df, text], axis=1)

#rinomino le colonne e droppo title e article
#df.rename(columns={0: 'text'}, inplace=True)
#df.drop(['title', 'article', 'Id'], inplace=True, axis=1)
#y = df['label']
#df.drop('label', inplace=True, axis=1)
df.columns

Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label'], dtype='object')

In [81]:
weekdays = []
daytime = []
for day in df['timestamp']:
    if day == '0000-00-00 00:00:00':
        week_day = -1
        hour_day = -1
    else:
        week_day = pd.Timestamp(day).day_of_week
        hour = pd.Timestamp(day).hour
        if hour > 5 and hour <= 11:
            hour_day = 2        #morning
        elif hour > 11 and hour <= 14:
            hour_day = 3        #lunch
        elif hour > 14 and hour <= 19:
            hour_day = 4        #afternoon
        elif hour > 19 and hour <= 21:
            hour_day = 5        #dinner
        else:
            hour_day = 1        #night

    daytime.append(hour_day)
    weekdays.append(week_day)

In [82]:
import re
import string
from bs4 import BeautifulSoup

def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'\d+', '', text) 
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

for i, article in zip(df.index, df['article']):
    df.loc[i, 'article'] = clean_text(article)

for i, title in zip(df.index, df['title']):
    df.loc[i, 'title'] = clean_text(title)

In [83]:
df['day_of_week'] = weekdays
df['moment_of_day'] = daytime

df['article_word_count'] = df['article'].apply(lambda x: len(str(x).split()))
df.drop('timestamp', axis=1, inplace=True)


In [84]:
mask = (df['article'] == 'n')
df[mask].groupby('label').count()

,Id,source,title,article,page_rank,day_of_week,moment_of_day,article_word_count
label,,,,,,,,
0,21,21,21,21,21,21,21,21
1,887,887,887,887,887,887,887,887
2,3,3,3,3,3,3,3,3
3,9,9,9,9,9,9,9,9
4,1,1,1,1,1,1,1,1
5,952,952,952,952,952,952,952,952


In [85]:
to_drop = df[mask].index
df.drop(to_drop, axis=0, inplace=True)


In [86]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [87]:
df['text'] = df['title'] + ' ' + df['title'] + ' ' + df['article']

In [88]:
df['article_word_count'] = df['text'].apply(lambda x: len(str(x).split()))
short_texts = df[df['article_word_count']  < 7]

print(short_texts['text'])

278                                                       
10507         sports briefing sports briefing pro football
14437               sports briefing sports briefing hockey
17263                           the churn the churn people
27219                       addenda addenda general motors
29661             fame fame on  applecom  newest trailers 
32244    russert contradicts libby russert contradicts ...
32426        snowstorm hits plains snowstorm hits plains  
34857                the perfect steak the perfect steak  
35346                         com   com   huaweis on first
39835      horror horror on  blogtelevision latest videos 
42981                       letters letters the skinny man
46714    iranresolution verschoben iranresolution versc...
53221    painkiller patches recalled painkiller patches...
54658        fed auctions  billion fed auctions  billion  
61379      horror horror on  blogtelevision latest videos 
76269         when worlds collide   when worlds collide 

In [89]:

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

In [90]:
X_train

,Id,source,title,article,page_rank,day_of_week,moment_of_day,article_word_count,text
14272,14272,BBC,ira not stalemate problem,the ira is not the problem in the negotiations...,5,-1,-1,26,ira not stalemate problem ira not stalemate pr...
27460,27460,Boston,photos of abuse in iraq shock british,london graphic photographs of british soldier...,5,-1,-1,45,photos of abuse in iraq shock british photos o...
58286,58286,Reuters,market slips as housing worries weigh,new york reuters stocks edged lower on friday...,5,4,5,35,market slips as housing worries weigh market s...
217,217,Yahoo,protesters aim to shut down british power stat...,afp hundreds of protesters have marched o...,5,3,4,48,protesters aim to shut down british power stat...
20973,20973,CNET,deal or no deal hddvd player cracks barrier,the price for toshibas hda hd dvd player has d...,3,1,2,33,deal or no deal hddvd player cracks barrier d...
...,...,...,...,...,...,...,...,...,...
42853,42853,Yahoo,british opposition party under fire for pledgi...,afp the leader of britains main opposition pa...,5,0,4,51,british opposition party under fire for pledgi...
36364,36364,Yahoo,jane is fiercely independent character ap,ap on kingdom mountain houghton mifflin page...,5,4,4,51,jane is fiercely independent character ap...
36215,36215,New,levens keeps cool head,jacksonville fla dorsey levens remembers well...,5,-1,-1,64,levens keeps cool head levens keeps cool head ...
55017,55017,RedNova,rain old sewers make for nasty story,philadelphia theres no way to tiptoe around t...,3,2,5,59,rain old sewers make for nasty story rain old ...


In [91]:
stop_words = list(ENGLISH_STOP_WORDS) + [
    # --- Rumore HTML e Scraping ---
    'http', 'https', 'src', 'href', 'img', 'alignleft', 'alignright', 
    'height', 'width', 'border', 'clearall', 'borderimga', 'br', 'class',
    
    # --- Agenzie Stampa e Formattazione News ---
    'reuters', 'ap', 'afp', 'pa', 'upi', 'press', 'association',
    'new', 'york', 'london', 'washington', # Città ricorrenti nelle dateline
    'said', 'says', 'told', 'reported', 'added', # Verbi di reportistica
    
    # --- Temporali Generici (inutili per classificare l'argomento) ---
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'yesterday', 'today', 'tomorrow', 'year', 'years', 'week', 'month', 'day',
    'time', 'daily', 'date']

vectorizer = TfidfVectorizer(
        max_features=30000,
        stop_words=stop_words,
        ngram_range=(1, 2),
        max_df=0.7,
        min_df=3,)


In [92]:

prep = ColumnTransformer(transformers=[
    ('text', vectorizer, 'text')],
    remainder='drop'
)

df_prep = prep.fit_transform(X_train)

In [93]:
text_vec = pd.DataFrame(df_prep.todense(), index=y_train.index)
text_vec['label'] = y_train.copy()

In [94]:
text_vec['label']

14272    5
27460    0
58286    1
217      0
20973    2
        ..
42853    0
36364    3
36215    4
55017    2
7184     0
Name: label, Length: 62497, dtype: int64

In [95]:
grouped = text_vec.groupby('label').mean()

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def getDis(X_sparse):
    # 1. Recupera la matrice SPARSA originale (quella uscita dal Vectorizer)
    # NON usare il DataFrame denso (text_vec) che ti impalla la RAM.

    # 2. Prepara i centroidi (assicurati che siano solo le features, senza label)
    # Se 'grouped' è un DataFrame coi centroidi:
    centroids_matrix = grouped.values[:, :30000] # O qualunque sia la dimensione delle features

    # 3. IL CALCOLO MAGICO (Tutto in una volta)
    # Restituisce una matrice di forma (62237, 7)
    # Righe = Articoli, Colonne = Similarità con i 7 centroidi
    similarity_matrix = cosine_similarity(X_sparse, centroids_matrix)

    # 4. Ora hai tutto. Vuoi riempire il tuo dizionario 'distances'?
    # È immediato: la colonna 0 contiene tutte le distanze dal centroide 0
    distances = {
        j: similarity_matrix[:, j].tolist() 
        for j in range(7)
    }
    print("Calcolo completato.")
    return distances


In [ ]:
distances = getDis(df_prep)
dist = pd.DataFrame(distances, index=X_train.index)
#dist['fake_label'] = np.argmax(dist,axis=1)
#print((dist['fake_label'] == y_train).sum())
#print(len(y_train))

Calcolo completato.


In [ ]:
X_train = pd.concat([X_train, dist], axis=1)

In [ ]:
X_train.columns = X_train.columns.astype(str)

In [ ]:
stop_words = list(ENGLISH_STOP_WORDS) + [
    # --- Rumore HTML e Scraping ---
    'http', 'https', 'src', 'href', 'img', 'alignleft', 'alignright', 
    'height', 'width', 'border', 'clearall', 'borderimga', 'br', 'class',
    
    # --- Agenzie Stampa e Formattazione News ---
    'reuters', 'ap', 'afp', 'pa', 'upi', 'press', 'association',
    'new', 'york', 'london', 'washington', # Città ricorrenti nelle dateline
    'said', 'says', 'told', 'reported', 'added', # Verbi di reportistica
    
    # --- Temporali Generici (inutili per classificare l'argomento) ---
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'yesterday', 'today', 'tomorrow', 'year', 'years', 'week', 'month', 'day',
    'time', 'daily', 'date']


encoder = OneHotEncoder(min_frequency=10, handle_unknown='infrequent_if_exist')

scaler = StandardScaler()

text_pipeline_SVD = Pipeline([
    ('vect', TfidfVectorizer(
        max_features=50000,
        stop_words=stop_words,
        ngram_range=(1, 2),
        max_df=0.8,
        min_df=3
    )),
    ('svd', TruncatedSVD(n_components=100))
])

vectorizer = TfidfVectorizer(
        max_features=30000,
        stop_words=stop_words,
        ngram_range=(1, 2),
        max_df=0.7,
        min_df=3,)


In [ ]:
# 1. Definisci quante parole vuoi per ogni classe
# Esempio: 50.000 totali diviso 7 classi = ~7.000 parole per classe
TOP_K_PER_CLASS = 7000 

final_vocabulary = set() # Usiamo un set per evitare duplicati (parole comuni a più classi)

# 2. Ciclo su ogni classe per estrarre le sue "parole d'oro"
print("Estrazione vocabolario specifico per classe...")
classes = np.unique(y_train)

for label in classes:
    # Prendi solo il testo di QUESTA classe
    subset_text = X_train[y_train == label]['text']
    
    # Fitta un vectorizer piccolo SOLO su questa classe
    temp_vectorizer = TfidfVectorizer(
        max_features=TOP_K_PER_CLASS,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2 # Qui puoi essere più permissivo perché siamo nel subset
    )
    temp_vectorizer.fit(subset_text)
    
    # Aggiungi le parole trovate al calderone comune
    words = temp_vectorizer.get_feature_names_out()
    final_vocabulary.update(words)
    print(f"Classe {label}: trovate {len(words)} parole specifiche.")

# 3. Converti il set in lista per passarlo a Sklearn
final_vocabulary_list = list(final_vocabulary)
print(f"Vocabolario totale ottimizzato: {len(final_vocabulary_list)} parole uniche.")

# 4. Crea il Vettorizzatore "Frankenstein" Finale
# Invece di max_features, passiamo 'vocabulary=final_vocabulary_list'
# Questo obbliga il vectorizer a cercare SOLO ed ESCLUSIVAMENTE quelle parole
master_vectorizer = TfidfVectorizer(
    vocabulary=final_vocabulary_list,  # <--- IL TRUCCO È QUI
    stop_words='english',
    ngram_range=(1, 2)
)
preprocessor = ColumnTransformer(transformers=[
    ('text', text_pipeline_SVD, 'text'),
    ('source', encoder, ['source']),
    ('numerical', scaler, ['0', '1', '2', '3', '4', '5', '6'])
], remainder='drop')


Estrazione vocabolario specifico per classe...
Classe 0: trovate 7000 parole specifiche.
Classe 1: trovate 7000 parole specifiche.
Classe 2: trovate 7000 parole specifiche.
Classe 3: trovate 7000 parole specifiche.
Classe 4: trovate 7000 parole specifiche.
Classe 5: trovate 7000 parole specifiche.
Classe 6: trovate 7000 parole specifiche.
Vocabolario totale ottimizzato: 23526 parole uniche.


In [ ]:
X_train = preprocessor.fit_transform(X_train)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat


In [ ]:
X_copy = prep.transform(X_test)

In [ ]:
vec_test = pd.DataFrame(X_copy.todense(), index=y_test.index)


distances = getDis(X_copy)
dist = pd.DataFrame(distances, index=y_test.index)

Calcolo completato.


In [ ]:

X_test = pd.concat([X_test, dist], axis=1)

In [ ]:
X_test.columns = X_test.columns.astype(str)

X_test = preprocessor.transform(X_test)

In [ ]:
weights = compute_sample_weight(class_weight='balanced', y=y_train)


clf = XGBClassifier(
        n_estimators=1000,      
        learning_rate=0.1,     
        max_depth=6,             
        min_child_weight=1,
        gamma=0.1,           
        subsample=0.8,           
        colsample_bytree=0.8,     
        n_jobs=-1,               
        random_state=42,
        eval_metric='mlogloss'  
    )
clf4 = LinearSVC(C=0.01,
                 class_weight='balanced',
                 tol=1e-10,
                 max_iter=1000)

clf4.fit(X_train, y_train)

y_pred = clf4.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.68      0.70      4681
           1       0.68      0.72      0.70      1933
           2       0.76      0.79      0.78      2227
           3       0.58      0.47      0.52      1984
           4       0.70      0.91      0.79      1712
           5       0.52      0.41      0.46      2404
           6       0.49      0.81      0.61       618

    accuracy                           0.66     15559
   macro avg       0.64      0.69      0.65     15559
weighted avg       0.66      0.66      0.66     15559

